# LABORATORIO DIRIGIDO N.° 03 — SQL avanzado
**Semana 03 · Fundamentos de Gestión de Datos · TECSUP 2026-II**  
**Caso:** TecnoAndina Retail — SQLite, `JOIN`, `GROUP BY`, `HAVING`, subconsultas y vistas.  
**Docente:** Pilar Rocío Sayán Mejía.  
**Equipo:** completar integrantes y sección antes de entregar.  
**Fuente de datos:** conjunto didáctico `LD3_TecnoAndina_Retail.xlsx`, disponible en el repositorio de la docente: https://github.com/Rociosayan/Repo_FDG_2026-2/tree/main/semana03/datos

> **Nota de procedencia:** notebook elaborado a partir de una copia de trabajo del Laboratorio D3 aportada para esta actividad. Se han corregido las respuestas, completado los retos y conservado las salidas SQL visibles de esa copia. Las salidas corresponden al dataset simulado de la guía y deben contrastarse al ejecutar **Entorno de ejecución → Ejecutar todas** en Google Colab. No son datos comerciales reales.


## Caso de estudio: TecnoAndina Retail

TecnoAndina Retail es una cadena peruana de tecnología con tiendas en Lima, Arequipa, Trujillo y Piura y un canal de venta por Web. Su información está repartida en cuatro tablas y el área comercial necesita saber qué clientes compran más, cuáles todavía no compran y qué productos y categorías sustentan las ventas.

Naturaleza de los datos. Conjunto de práctica inventado (no son ventas ni personas reales), con pedidos del 1 de agosto al 30 de septiembre de 2026, publicado en GitHub en LD3_TecnoAndina_Retail.xlsx. Sirve para practicar consultas, no para afirmar tendencias ni rentabilidad.

- clientes: id_cliente, nombre, ciudad. Una fila es un cliente registrado.

- productos: id_producto, nombre_producto, categoria, precio_lista. Una fila es un producto del catálogo.

- pedidos: id_pedido, id_cliente, fecha_pedido, canal, total. Una fila es un pedido de un cliente.

- detalles_pedido: id_detalle, id_pedido, id_producto, cantidad, precio_unitario. Una fila es un producto incluido en un pedido.

Las tablas se relacionan por id_cliente (clientes–pedidos), id_pedido (pedidos–detalles_pedido) e id_producto (productos–detalles_pedido).

### Preparación del entorno

Abra el notebook en Google Colab y ejecute la celda 1: importa sqlite3 y pandas, crea la base en memoria, carga las cuatro hojas del Excel desde GitHub y cuenta los registros. Cada resultado se muestra con pd.read_sql_query; pandas muestra los NULL como NaN o None. Si reinicia el entorno, vuelva a ejecutar la celda 1.

**Celda 1 de 12**

In [ ]:
import sqlite3
import pandas as pd

# Base de práctica de TecnoAndina Retail (agosto y septiembre de 2026)
url = "https://raw.githubusercontent.com/Rociosayan/Repo_FDG_2026-2/main/semana03/datos/LD3_TecnoAndina_Retail.xlsx"
conexion = sqlite3.connect(":memory:")

# Cada hoja del Excel pasa a una tabla de SQLite
pd.read_excel(url, sheet_name="clientes").to_sql("clientes", conexion, index=False)
pd.read_excel(url, sheet_name="productos").to_sql("productos", conexion, index=False)
pd.read_excel(url, sheet_name="pedidos").to_sql("pedidos", conexion, index=False)
pd.read_excel(url, sheet_name="detalles_pedido").to_sql("detalles_pedido", conexion, index=False)

consulta = """SELECT
  (SELECT COUNT(*) FROM clientes) AS clientes,
  (SELECT COUNT(*) FROM productos) AS productos,
  (SELECT COUNT(*) FROM pedidos) AS pedidos,
  (SELECT COUNT(*) FROM detalles_pedido) AS detalles_pedido;"""
pd.read_sql_query(consulta, conexion)

,clientes,productos,pedidos,detalles_pedido
0,24,14,42,68


## A. Relacionar tablas

### Ejercicio 1 — INNER JOIN y LEFT JOIN: clientes con y sin pedidos

El área comercial quiere saber cuántos clientes compraron y quiénes todavía no registran pedidos. INNER JOIN conserva solo las filas con pareja; LEFT JOIN conserva todas las filas de la tabla izquierda y completa con NULL.

Se solicita. Ejecute las celdas 2 y 3.

**Celda 2 de 12**

In [ ]:
consulta = """SELECT COUNT(p.id_pedido) AS pedidos_unidos,
       COUNT(DISTINCT c.id_cliente) AS clientes_con_pedidos,
       (SELECT COUNT(*) FROM clientes) AS clientes_registrados
FROM pedidos AS p
INNER JOIN clientes AS c ON p.id_cliente = c.id_cliente;"""
pd.read_sql_query(consulta, conexion)

,pedidos_unidos,clientes_con_pedidos,clientes_registrados
0,42,18,24


**Celda 3 de 12**

In [ ]:
consulta = """SELECT c.id_cliente, c.nombre, c.ciudad
FROM clientes AS c
LEFT JOIN pedidos AS p ON c.id_cliente = p.id_cliente
WHERE p.id_pedido IS NULL
ORDER BY c.id_cliente;"""
pd.read_sql_query(consulta, conexion)

,id_cliente,nombre,ciudad
0,6,Andrés Villanueva,Lima
1,11,Raúl Benavides,Arequipa
2,15,Héctor Burga,Trujillo
3,20,Marta Infante,Piura
4,22,Gabriela Ponce,Arequipa
5,24,Natalia Ato,Piura


### Respuestas a los ejercicios 1 y 2

**Pregunta 1.** ¿Cuántos pedidos y clientes distintos devuelve el `INNER JOIN`? ¿Por qué no aparecen todos los clientes?

**Respuesta.** La celda 2 devuelve **42 pedidos vinculados** a **18 clientes diferentes**, frente a **24 clientes registrados**. `INNER JOIN` exige coincidencia entre `pedidos.id_cliente` y `clientes.id_cliente`; por eso excluye a los **6 clientes sin pedidos**. `COUNT(p.id_pedido)` cuenta pedidos, mientras que `COUNT(DISTINCT c.id_cliente)` cuenta personas distintas de la tabla `clientes`.

**Pregunta 2.** ¿Cuál es la tabla izquierda y por qué se usa `IS NULL`?

**Respuesta.** `clientes AS c` es la tabla izquierda. `LEFT JOIN` conserva sus 24 clientes, incluso los que no tienen registros relacionados en `pedidos`. El filtro `WHERE p.id_pedido IS NULL` identifica exclusivamente a los **6 clientes sin pedidos**: Andrés Villanueva (Lima), Raúl Benavides y Gabriela Ponce (Arequipa), Héctor Burga (Trujillo), Marta Infante y Natalia Ato (Piura). Se usa `IS NULL` y no `= NULL`, ya que en SQL la comparación de un valor con `NULL` mediante `=` produce un resultado desconocido, no verdadero.

### Ejercicio 2 — `RIGHT JOIN` y `FULL OUTER JOIN`

El área de Compras identifica productos nunca incluidos en pedidos y compara clientes con compras en agosto y septiembre. `RIGHT JOIN` conserva toda la tabla derecha y `FULL OUTER JOIN` conserva los registros presentes en cualquiera de ambos resúmenes. Requieren SQLite **3.39.0 o superior**.

**Se solicita:** ejecutar las celdas 4 y 5 y analizar la presencia de `NULL` en las columnas de cada mes.


**Celda 4 de 12**

In [ ]:
# RIGHT JOIN y FULL OUTER JOIN requieren SQLite 3.39.0 o posterior
consulta = """SELECT sqlite_version() AS version_sqlite;"""
display(pd.read_sql_query(consulta, conexion))

consulta = """SELECT pr.id_producto, pr.nombre_producto, pr.categoria
FROM detalles_pedido AS d
RIGHT JOIN productos AS pr ON d.id_producto = pr.id_producto
WHERE d.id_detalle IS NULL
ORDER BY pr.id_producto;"""
display(pd.read_sql_query(consulta, conexion))

,version_sqlite
0,3.45.1


,id_producto,nombre_producto,categoria
0,5,Impresora multifuncional,Cómputo
1,14,Reloj inteligente,Accesorios


**Celda 5 de 12**

In [ ]:
conexion.executescript("""
DROP VIEW IF EXISTS resumen_agosto;
DROP VIEW IF EXISTS resumen_septiembre;
CREATE TEMP VIEW resumen_agosto AS
SELECT id_cliente, COUNT(id_pedido) AS num_pedidos, SUM(total) AS monto
FROM pedidos
WHERE fecha_pedido BETWEEN '2026-08-01' AND '2026-08-31'
GROUP BY id_cliente;
CREATE TEMP VIEW resumen_septiembre AS
SELECT id_cliente, COUNT(id_pedido) AS num_pedidos, SUM(total) AS monto
FROM pedidos
WHERE fecha_pedido BETWEEN '2026-09-01' AND '2026-09-30'
GROUP BY id_cliente;
""")
consulta = """SELECT COALESCE(a.id_cliente, s.id_cliente) AS id_cliente,
       a.monto AS monto_agosto, s.monto AS monto_septiembre
FROM resumen_agosto AS a
FULL OUTER JOIN resumen_septiembre AS s ON a.id_cliente = s.id_cliente
ORDER BY id_cliente;"""
pd.read_sql_query(consulta, conexion)

,id_cliente,monto_agosto,monto_septiembre
0,1,1435.0,1967.0
1,2,1522.0,2615.0
2,3,656.0,840.0
3,4,2452.0,NaN
4,5,NaN,640.0
5,7,165.0,3460.0
6,8,271.0,236.0
7,9,220.0,NaN
8,10,6400.0,723.0
9,12,NaN,1420.0


**Pregunta 3.** ¿Qué versión de SQLite informó la celda 4? ¿Qué tabla es la derecha y cuáles son los productos sin ventas?

**Respuesta.** La salida conservada informa **SQLite 3.45.1**. En `detalles_pedido AS d RIGHT JOIN productos AS pr`, la tabla derecha es **`productos`**; todas sus filas se mantienen aunque no haya coincidencias. El filtro `WHERE d.id_detalle IS NULL` muestra los productos sin detalle vendido: **ID 5: Impresora multifuncional (Cómputo)** y **ID 14: Reloj inteligente (Accesorios)**. La versión puede cambiar al ejecutar Colab hoy, aunque debe ser 3.39.0 o mayor.

**Pregunta 4.** ¿Cuántos clientes compraron solo en agosto, solo en septiembre y en ambos meses? ¿Cómo reconocerlos por los `NULL`?

**Respuesta.** **3 solo en agosto** (ID 4, 9 y 13), porque `monto_septiembre` es `NULL`; **3 solo en septiembre** (ID 5, 12 y 18), porque `monto_agosto` es `NULL`; y **12 en ambos meses**, porque ambas columnas tienen montos numéricos. En total son **18 clientes compradores distintos**. `NULL` indica ausencia de compras de ese cliente en ese mes, no una venta de S/ 0.

## B. Agrupar y filtrar grupos

### Ejercicio 3 — `WHERE`, `GROUP BY` y `HAVING`

Se identifican los clientes con **al menos dos pedidos realizados en septiembre**. `WHERE` se aplica antes de agrupar y `HAVING` se aplica después de formar los grupos.


**Celda 6 de 12**

In [ ]:
consulta = """SELECT c.id_cliente, c.nombre,
       COUNT(p.id_pedido) AS pedidos_septiembre,
       SUM(p.total) AS monto_septiembre
FROM clientes AS c
INNER JOIN pedidos AS p ON c.id_cliente = p.id_cliente
WHERE p.fecha_pedido >= '2026-09-01'
GROUP BY c.id_cliente, c.nombre
HAVING COUNT(p.id_pedido) >= 2
ORDER BY monto_septiembre DESC, c.id_cliente;"""
pd.read_sql_query(consulta, conexion)

,id_cliente,nombre,pedidos_septiembre,monto_septiembre
0,2,Carlos Mendoza,3,2615
1,1,Lucía Paredes,2,1967
2,12,Sofía Linares,2,1420
3,17,Luis Távara,2,885
4,10,Patricia Zevallos,2,723


**Pregunta 5.** ¿Qué filtra `WHERE` y qué filtra `HAVING`? Si un cliente tuvo tres pedidos en total, pero solo uno en septiembre, ¿aparece?

**Respuesta.** `WHERE p.fecha_pedido >= '2026-09-01'` conserva las filas correspondientes a septiembre dentro del periodo simulado; `GROUP BY` agrupa por cliente y `HAVING COUNT(p.id_pedido) >= 2` deja solo grupos con **dos o más pedidos de septiembre**. No aparece un cliente con tres pedidos históricos si **solo uno** ocurrió en septiembre. La salida muestra **5 clientes**: Carlos Mendoza (ID 2, 3 pedidos, S/ 2 615), Lucía Paredes (ID 1, 2, S/ 1 967), Sofía Linares (ID 12, 2, S/ 1 420), Luis Távara (ID 17, 2, S/ 885) y Patricia Zevallos (ID 10, 2, S/ 723).

## C. Consultar y reutilizar resultados

### Ejercicio 4 — Subconsulta

Se comparan pedidos individuales con el promedio de **todos los pedidos**, sin confundirlo con el promedio del monto acumulado por cliente.


**Celda 7 de 12**

In [ ]:
consulta = """SELECT p.id_pedido, p.fecha_pedido, c.id_cliente, c.nombre, p.total
FROM pedidos AS p
INNER JOIN clientes AS c ON p.id_cliente = c.id_cliente
WHERE p.total > (SELECT AVG(total) FROM pedidos)
ORDER BY p.total DESC, p.id_pedido;"""
pd.read_sql_query(consulta, conexion)

,id_pedido,fecha_pedido,id_cliente,nombre,total
0,3,2026-08-02,10,Patricia Zevallos,6400
1,13,2026-08-17,13,Diego Alva,5655
2,26,2026-09-08,7,Carlos Mendoza,3460
3,31,2026-09-15,18,Ana Girón,3045
4,35,2026-09-24,2,Carlos Mendoza,2544
5,20,2026-08-21,4,Jorge Salazar,2452
6,9,2026-08-12,16,Camila Neyra,2256
7,15,2026-08-18,17,Luis Távara,1680
8,11,2026-08-16,2,Carlos Mendoza,1522
9,28,2026-09-10,1,Lucía Paredes,1485


**Pregunta 7.** ¿Qué devuelve la subconsulta, cuántos pedidos la superan y qué clientes aparecen más de una vez?

**Respuesta.** `(SELECT AVG(total) FROM pedidos)` devuelve el **monto promedio por pedido**, aproximadamente **S/ 1 030,36** (S/ 43 275 ÷ 42 pedidos). La consulta exterior devuelve **13 pedidos** superiores a ese promedio. Entre ellos, **Lucía Paredes, ID 1, aparece dos veces** (S/ 1 485 y S/ 1 240) y **Carlos Mendoza, ID 2, aparece dos veces** (S/ 2 544 y S/ 1 522). También aparece otro **Carlos Mendoza, ID 7**, por una compra de S/ 3 460; no debe confundirse con el cliente ID 2 aunque tenga el mismo nombre.

### Ejercicio 5 — Vista reutilizable

La vista resume por identificador de cliente el número de pedidos, el monto acumulado y el promedio de cada pedido.


**Celda 8 de 12**

In [ ]:
conexion.executescript("""
DROP VIEW IF EXISTS vista_resumen_cliente;
CREATE VIEW vista_resumen_cliente AS
SELECT c.id_cliente, c.nombre, c.ciudad,
       COUNT(p.id_pedido) AS num_pedidos,
       COALESCE(SUM(p.total), 0) AS monto_acumulado,
       ROUND(AVG(p.total), 2) AS promedio_pedido
FROM clientes AS c
LEFT JOIN pedidos AS p ON c.id_cliente = p.id_cliente
GROUP BY c.id_cliente, c.nombre, c.ciudad;
""")
consulta = """SELECT *
FROM vista_resumen_cliente
WHERE nombre = 'Carlos Mendoza' OR ciudad = 'Piura'
ORDER BY ciudad, monto_acumulado DESC, id_cliente;"""
pd.read_sql_query(consulta, conexion)

,id_cliente,nombre,ciudad,num_pedidos,monto_acumulado,promedio_pedido
0,7,Carlos Mendoza,Arequipa,2,3625,1812.50
1,2,Carlos Mendoza,Lima,4,4137,1034.25
2,17,Luis Távara,Piura,4,3224,806.00
3,18,Ana Girón,Piura,1,3045,3045.00
4,19,Pedro Seminario,Piura,2,1502,751.00
5,20,Marta Infante,Piura,0,0,NaN
6,24,Natalia Ato,Piura,0,0,NaN


**Pregunta 8.** ¿Qué guarda la vista y qué distingue a los dos clientes llamados Carlos Mendoza?

**Respuesta.** `CREATE VIEW` guarda **la definición de una consulta SQL**, no una copia fija de sus resultados. La vista se evalúa contra las tablas al consultarla. El campo **`id_cliente`** distingue a Carlos Mendoza de **Lima (ID 2, S/ 4 137, cuatro pedidos)** de Carlos Mendoza de **Arequipa (ID 7, S/ 3 625, dos pedidos)**. Agrupar solo por `nombre` mezclaría indebidamente personas distintas.

## Actividad 3 — Informe analítico comparado

El reporte siguiente ordena a los **cinco clientes con mayor monto acumulado** de los pedidos de agosto y septiembre de 2026. El total se agrega **una sola vez por pedido** desde `pedidos`, y no después de combinarlo con `detalles_pedido`, porque esa unión repetiría `p.total` por cada línea de detalle.


**Celda 9 de 12**

In [ ]:
# Clientes con mayor monto acumulado (top cinco)
consulta = """SELECT c.id_cliente, c.nombre, c.ciudad,
       COUNT(p.id_pedido) AS num_pedidos,
       SUM(p.total) AS monto_acumulado,
       ROUND(AVG(p.total), 2) AS promedio_pedido
FROM clientes AS c
INNER JOIN pedidos AS p ON c.id_cliente = p.id_cliente
GROUP BY c.id_cliente, c.nombre, c.ciudad
ORDER BY monto_acumulado DESC, c.id_cliente
LIMIT 5;"""
pd.read_sql_query(consulta, conexion)

,id_cliente,nombre,ciudad,num_pedidos,monto_acumulado,promedio_pedido
0,10,Patricia Zevallos,Arequipa,3,7123,2374.33
1,13,Diego Alva,Trujillo,2,5879,2939.50
2,2,Carlos Mendoza,Lima,4,4137,1034.25
3,7,Carlos Mendoza,Arequipa,2,3625,1812.50
4,1,Lucía Paredes,Lima,4,3402,850.50


### Top 5 de clientes por monto acumulado

| Posición | ID | Cliente | Ciudad | Pedidos | Monto acumulado (S/) | Promedio por pedido (S/) |
|---:|---:|---|---|---:|---:|---:|
| 1 | 10 | Patricia Zevallos | Arequipa | 3 | 7 123,00 | 2 374,33 |
| 2 | 13 | Diego Alva | Trujillo | 2 | 5 879,00 | 2 939,50 |
| 3 | 2 | Carlos Mendoza | Lima | 4 | 4 137,00 | 1 034,25 |
| 4 | 7 | Carlos Mendoza | Arequipa | 2 | 3 625,00 | 1 812,50 |
| 5 | 1 | Lucía Paredes | Lima | 4 | 3 402,00 | 850,50 |

**A. ¿Quién lidera por monto acumulado y quién tiene el mayor promedio por pedido?**  
**Respuesta.** **Patricia Zevallos (ID 10)** encabeza el monto acumulado con **S/ 7 123** repartidos en tres pedidos. **Diego Alva (ID 13)** presenta el **mayor promedio por pedido, S/ 2 939,50**, con dos pedidos y S/ 5 879 acumulados. Los dos clientes llamados Carlos Mendoza son personas distintas (ID 2 e ID 7).

**B. ¿Qué datos adicionales se necesitan antes de conceder beneficios? Recomendación prudente.**  
**Respuesta.** Solicitaría **costos, márgenes reales, devoluciones, descuentos, historial de compras más amplio, antigüedad, recurrencia e impacto previsto de incentivos**. Recomendaría evaluar un piloto de beneficios segmentado y medir su margen incremental, no premiar automáticamente a quien factura más. **Ingresos altos no equivalen necesariamente a mayor rentabilidad**; agosto y septiembre de 2026 son datos ficticios y no demuestran tendencias generales.

## Reto de aplicación y retroalimentación

Cada reto utiliza consultas SQL que deben quedar visibles junto con su salida y su explicación técnica.


**Celda 10 de 12**

In [ ]:
consulta = """SELECT c.ciudad, COUNT(c.id_cliente) AS clientes_sin_pedidos
FROM clientes AS c
LEFT JOIN pedidos AS p ON c.id_cliente = p.id_cliente
WHERE p.id_pedido IS NULL
GROUP BY c.ciudad
ORDER BY clientes_sin_pedidos DESC, c.ciudad;"""
pd.read_sql_query(consulta, conexion)

,ciudad,clientes_sin_pedidos
0,Arequipa,2
1,Piura,2
2,Lima,1
3,Trujillo,1


**Pregunta de cierre.** ¿Cuántos clientes sin pedidos hay y dónde se concentran? ¿Qué cláusula los conserva y cuál los aísla?

**Respuesta.** Hay **6 clientes sin pedidos**: **2 en Arequipa, 2 en Piura, 1 en Lima y 1 en Trujillo**. `LEFT JOIN` conserva todos los clientes, `WHERE p.id_pedido IS NULL` aísla los que no tienen coincidencia, `GROUP BY c.ciudad` cuenta por ciudad y `ORDER BY` ordena el resultado. No se puede concluir que esos clientes estén perdidos: solo no tienen pedidos en el periodo simulado.

### Reto 2 — Ingresos por categoría

El ingreso se obtiene de `SUM(cantidad * precio_unitario)` sobre los detalles, sin sumar indebidamente el `total` del pedido después de unirlo con sus varias líneas.


**Celda 11 de 12**

In [ ]:
consulta = """SELECT pr.categoria,
       SUM(d.cantidad) AS unidades,
       SUM(d.cantidad * d.precio_unitario) AS ingreso
FROM productos AS pr
INNER JOIN detalles_pedido AS d ON pr.id_producto = d.id_producto
GROUP BY pr.categoria
HAVING SUM(d.cantidad * d.precio_unitario) >= 10000
ORDER BY ingreso DESC, pr.categoria;"""
pd.read_sql_query(consulta, conexion)

,categoria,unidades,ingreso
0,Celulares,12,18805
1,Cómputo,37,17662


**Pregunta de cierre.** ¿Qué categorías superan S/ 10 000? ¿Cuál queda fuera y por qué se usa `HAVING`?

**Respuesta.** Según la salida aparecen **Celulares: 12 unidades, S/ 18 805** y **Cómputo: 37 unidades, S/ 17 662**. Las demás categorías **no superaron el umbral o no tuvieron ventas**; por ejemplo, el producto de **Accesorios** identificado como no vendido en la celda 4 no aportó ingreso. Para conocer todos los nombres y totales de las categorías excluidas debe ejecutarse la misma consulta **sin `HAVING`**. `HAVING SUM(d.cantidad * d.precio_unitario) >= 10000` filtra **grupos agregados**; `WHERE` no puede evaluar directamente ese agregado sobre el grupo.

### Reto 3 — Clientes sobre el promedio por cliente

La subconsulta calcula el promedio del **monto acumulado por cliente comprador** (`num_pedidos > 0`), distinto del promedio por pedido calculado en el ejercicio 4.


**Celda 12 de 12**

In [ ]:
consulta = """SELECT id_cliente, nombre, ciudad, monto_acumulado
FROM vista_resumen_cliente
WHERE monto_acumulado > (SELECT AVG(monto_acumulado)
                         FROM vista_resumen_cliente
                         WHERE num_pedidos > 0)
ORDER BY monto_acumulado DESC, id_cliente;"""
pd.read_sql_query(consulta, conexion)

,id_cliente,nombre,ciudad,monto_acumulado
0,10,Patricia Zevallos,Arequipa,7123
1,13,Diego Alva,Trujillo,5879
2,2,Carlos Mendoza,Lima,4137
3,7,Carlos Mendoza,Arequipa,3625
4,1,Lucía Paredes,Lima,3402
5,17,Luis Távara,Piura,3224
6,18,Ana Girón,Piura,3045
7,16,Camila Neyra,Trujillo,2518
8,4,Jorge Salazar,Lima,2452


**Pregunta de cierre.** ¿Qué valor devuelve la subconsulta y cuántos clientes lo superan? ¿Por qué no compararlo con el promedio por pedido?

**Respuesta.** Devuelve **S/ 2 404,17** aproximadamente: S/ 43 275 de pedidos acumulados entre **18 clientes compradores**. Lo superan **9 clientes**, comenzando por Patricia Zevallos (S/ 7 123), Diego Alva (S/ 5 879), Carlos Mendoza de Lima (S/ 4 137) y Carlos Mendoza de Arequipa (S/ 3 625). El promedio por pedido es **S/ 1 030,36** (42 pedidos); son unidades de observación diferentes y no deben confundirse.

## Conclusiones del equipo

1. **Relaciones y cobertura.** `INNER JOIN` devuelve los 42 pedidos de 18 compradores, mientras que `LEFT JOIN` permite encontrar a los 6 clientes registrados sin pedidos. `RIGHT JOIN` permite identificar productos sin ventas y `FULL OUTER JOIN` muestra clientes activos en solo uno o en ambos meses.
2. **Agregaciones confiables.** `WHERE` filtra operaciones y `HAVING` filtra resultados agrupados. El top cinco se calcula sumando `pedidos.total` por `id_cliente`, **sin unir previamente las líneas de detalle**, lo que evita duplicar los totales. Patricia Zevallos encabeza el acumulado, pero Diego Alva tiene el promedio individual por pedido más alto.
3. **Decisiones y limitaciones.** Ni ingresos ni frecuencia demuestran rentabilidad. Los 24 clientes y 42 pedidos son **datos inventados de agosto–septiembre de 2026**; no autorizan generalizar tendencias, establecer preferencias definitivas ni recomendar beneficios sin márgenes, costes y datos históricos reales.

**Control de entrega:** el archivo `.ipynb` debe ubicarse en `semana03/` de un repositorio GitHub accesible. Abrirlo desde GitHub y verificar que se ven las 12 consultas, sus salidas, estas interpretaciones, el top cinco y los tres retos. Las cuatro preguntas del cuestionario individual no se incluyen porque no fueron proporcionadas en el material.
